In [1]:
import os

import importlib
import json
import pandas as pd
import random
import time

from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI

from pathlib import Path
import sys

In [2]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

import parsing
import evaluation
import exporting
import corpus
import error_sampling
import api

from dev import rel, print_epi_summary
from data_config import DATA_CONFIG, FEW_SHOT_PATH
from few_shot import get_few_shot_examples

import pipeline

# Config.

### Development Config.

In [ ]:
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at2026-08-08 21:40


### EPI Config.

In [4]:
# ---------- Manual EPI config. entry ----------
EPI_NUM = "005"
DATASET_SPLIT = "train"

# Corpus Loading

### Load Corpus

In [5]:
# ---------- Manually-set abstracts path ----------
ABSTRACTS_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["abstracts"]
SAMPLE_SIZE = DATA_CONFIG[DATASET_SPLIT]["sample_size"]

# ---------- Load corpus from path ----------
abstracts_corpus = corpus.load_corpus(ABSTRACTS_PATH, sample_size=SAMPLE_SIZE)
print(f"Abstracts dataset length: {len(abstracts_corpus)}")

Abstracts dataset length: 35


### Get Ground Truths
* Optional sample functionality

In [6]:
# ---------- Manually Set Ground Truth Path ----------
GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["ground_truths"]

# ---------- Fetch BioRED Ground Truths ----------
abstracts_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, GT_PATH)

In [7]:
# ---------- Turn ground truth DataFrame into structured dict ----------
ground_truths = corpus.get_gt_dict(abstracts_ground_truths)

### Few-Shot Construction

In [8]:
# ---------- If FS block does not exist, create FS block, else pass ----------
if not os.path.exists(FEW_SHOT_PATH):

    few_shot_block = get_few_shot_examples(
        path_to_train_set=DATA_CONFIG["train"]["paths"]["abstracts"],
        path_to_train_gts=DATA_CONFIG["train"]["paths"]["ground_truths"],
        biored_train_samples=abstracts_corpus,
        few_shot_export_path=FEW_SHOT_PATH
    )
    print(f"Generated and exported new few-shot block to '{FEW_SHOT_PATH}'")
else:
    with open(FEW_SHOT_PATH) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{FEW_SHOT_PATH}' at {datetime.now().strftime('%Y-%m-%d %H:%M')}.")

Imported existing few-shot block from '../../data/few_shot/few_shot_block.txt' at 2026-08-08 21:40.


### Import BioRED Extraction Guidelines

In [9]:
# ---------- Import BioRED guidelines text file for prompt refinement ----------
with open("../../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

# ---------- Print preview ----------
print(f"{biored_ext_guidelines[:500]}...")

## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover...


# OpenAI Luna API Call

### EPI Setup

In [10]:
# ---------- Create EPI setup dictionary ----------
# - Keys: "dataset", "id", "eval_version", "notes", "reuse_api_call", "prompt"
epi_setup = pipeline.setup_epi(EPI_NUM, DATASET_SPLIT)

# ---------- Print summary ----------
print_epi_summary(
    epi_num=EPI_NUM,
    prompt_version=epi_setup["prompt"]["version"],
    eval_version=epi_setup["eval_version"],
    notes=epi_setup["notes"],
    reuse_api_call=epi_setup["reuse_api_call"]
)

Cell ran at 2026-08-08 21:40 for epi_005
 - Prompt version: v3
 - Evaluation version: v3
 - Notes: Adds cosine similarity to performance eval.
 - Reuse API Call: True


### API Call

In [11]:
epi_setup.keys()

dict_keys(['dataset', 'id', 'eval_version', 'notes', 'reuse_api_call', 'prompt'])

In [12]:
# ---------- Create client ----------
client = api.create_client()

# ---------- If prompt version is different from previous EPI, call API, else pass ----------
if not epi_setup["reuse_api_call"]:

    print(f"Running API call for {epi_setup["id"]}...")

    # Fetch raw prompt template
    prompt_template = epi_setup["prompt"]["template"]

    # Begin timer
    start_time = time.perf_counter()

    # Initiate outputs list
    outputs = []

    # Begin looping through abstract dataset rows - one call per row
    for index, row in abstracts_corpus.iterrows():
        abstract = row["abstract"]

        # Replace prompt template's placeholders with abstract, few_shot & guidelines
        prompt = (
            prompt_template
            .replace("{abstract}", abstract)
            .replace("{few_shot_block}", few_shot_block)
            .replace("{biored_ext_guidelines}", biored_ext_guidelines)
        )

        # Store row's response
        response = client.responses.create(
            model="gpt-5.6-luna",
            input=prompt
        )

        # Append response to outputs list
        outputs.append({
            "pmid": row["pmid"],
            "output": response.output_text
        })

    # End time, store elapsed time & print result
    elapsed_seconds = time.perf_counter() - start_time
    print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts\n")

    print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f" - Prompt verion: {epi_setup["prompt"]["version"]}")
    print(f" - Evalaution verion: {epi_setup["eval_version"]}")
    print(f" - Run notes: {epi_setup["notes"]}")
    print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
else:
    prev_epi_id = f"epi_{int(EPI_NUM) - 1:03d}"

    with open(f"{DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]}/{prev_epi_id}.json") as f:
        prev_epi_log = json.load(f)

    outputs = prev_epi_log["outputs"]
    prompt_template = prev_epi_log["prompt"]
    elapsed_seconds = prev_epi_log["time_taken"]

    print(f"Reused API call from {prev_epi_id}.")

output_info = {
    "epi_id": epi_setup["id"],
    "outputs": outputs,
    "time_taken": elapsed_seconds,
    "raw_prompt": prompt_template,
    "epi_notes": epi_setup["notes"],
    "prompt_version": epi_setup["prompt"]["version"],
    "eval_version": epi_setup["eval_version"],
    "export_path": DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]
}

Created client:
 - Timeout: 60
 - Max retries: 0



Reused API call from epi_004.


In [ ]:
# ---------- Parse outputs, evaluate extractions, export EPI results ----------
epi_log = pipeline.process_epi(output_info, ground_truths, dataset=epi_setup["dataset"])

Parsed 35 extractions, 0 failed to parse as JSON


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Error Analysis

### Relations

In [ ]:
# ---------- False positives ----------
relations_fp = epi_log["errors"]["errors"]["relations"]["false_positives"]

print(f"Count: {len(relations_fp)}")
display(relations_fp)

Count: 275


[[15970799,
  'slco1b1*5',
  'Negative_Correlation',
  'estradiol-17beta-d-glucuronide'],
 [29222418, 'id3', 'Negative_Correlation', 'sox4'],
 [21163864, 'agt', 'Association', 'shcm'],
 [15099351, 'd374y', 'Association', 'hypercholesterolemia'],
 [19891556,
  'variant allele',
  'Positive_Correlation',
  'adverse cardiovascular events'],
 [15970799, 'slco1b1*5', 'Negative_Correlation', 'pravastatin'],
 [16120104, 'hper2', 'Association', 'g3853a'],
 [18808529, 'dystrophin', 'Bind', 'actin'],
 [20510337, 'coenzyme q10', 'Positive_Correlation', 'superoxide dismutase'],
 [17192049, 't3801c', 'Association', 'prostate cancer'],
 [18768591, 'nephrotic syndrome', 'Positive_Correlation', 'aldosterone'],
 [15970799, 'slco1b1*5', 'Negative_Correlation', 'cerivastatin'],
 [15099351, 'pcsk9', 'Positive_Correlation', 'familial hypercholesterolemia'],
 [19521089, "parkinson's disease", 'Association', 'resting tremor'],
 [20431083, 'mb', 'Positive_Correlation', 'warfarin'],
 [28428256, 'lps', 'Positiv

In [ ]:
# ---------- False negatives ----------
relations_fn = epi_log["errors"]["errors"]["relations"]["false_negatives"]

print(f"Count: {len(relations_fn)}\n")
display(relations_fn)

Count: 215



[[25305591, 'mog35-55', 'Positive_Correlation', 'vpac2'],
 [28428256, 'prostaglandin a2', 'Negative_Correlation', 'inflammatory'],
 [24914936, 'thyroxine', 'Association', 'skeletal dysplasia'],
 [25305591, 'foxp3', 'Positive_Correlation', 'vpac2'],
 [18827003,
  'aspartic acid to histidine substitution at amino acid position 401',
  'Positive_Correlation',
  'metabolic syndrome'],
 [24477591, '-930a>g', 'Positive_Correlation', 'coronary artery disease'],
 [25305591, 'vpac2', 'Positive_Correlation', 'anti-inflammatory cytokines'],
 [16737910, 'cancer', 'Association', 'etv6'],
 [25305591,
  'experimental autoimmune encephalomyelitis',
  'Negative_Correlation',
  'anti-inflammatory cytokines'],
 [16288197, 'congenital microcoria', 'Association', 'myocilin'],
 [10491763, 'hepatocyte nuclear factor-6', 'Association', 'type ii diabetes'],
 [28512644, 'ccl4', 'Association', 'erysipelas'],
 [16574712, 'memory deficits', 'Positive_Correlation', 'mdma'],
 [18768591,
  'serum- and glucocorticoid-

### Entities

In [ ]:
# ---------- False positives ----------
entities_fp = epi_log["errors"]["errors"]["entities"]["false_positives"]

print(f"Count: {len(entities_fp)}\n")
display(entities_fp)

Count: 206



[[10491763, 'c-peptide', 'ChemicalEntity'],
 [24914936, 'tsh', 'ChemicalEntity'],
 [24743235, 'il-1', 'ChemicalEntity'],
 [24743235, 'il-6', 'ChemicalEntity'],
 [19319147, 'pcc', 'ChemicalEntity'],
 [16288197, 'glaucoma', 'DiseaseOrPhenotypicFeature'],
 [24743235, 'csf-1', 'ChemicalEntity'],
 [21163864, 'hypertrophic cardiomyopathy', 'DiseaseOrPhenotypicFeature'],
 [24477591, 'cad', 'DiseaseOrPhenotypicFeature'],
 [28512644, 'erythematous erysipelas', 'DiseaseOrPhenotypicFeature'],
 [16288197, 'q48h', 'SequenceVariant'],
 [28512644, 'cat c262t', 'SequenceVariant'],
 [28428256, 'vasp', 'GeneOrGeneProduct'],
 [24341598, 'creatinine', 'ChemicalEntity'],
 [24341598, 'arf', 'DiseaseOrPhenotypicFeature'],
 [15970799, 'slco1b1*5', 'SequenceVariant'],
 [17192049, 'w2/m2', 'SequenceVariant'],
 [24341598, 'cin', 'DiseaseOrPhenotypicFeature'],
 [19521089, 'muscular rigidity', 'DiseaseOrPhenotypicFeature'],
 [24477591, 'rs9932581:a>g', 'SequenceVariant'],
 [10491763, 'hnf-6', 'GeneOrGeneProduct'],

In [ ]:
# ---------- False negatives ----------
entities_fn = epi_log["errors"]["errors"]["entities"]["false_negatives"]

print(f"Count: {len(entities_fn)}\n")
display(entities_fn)

Count: 46



[[25305591, 'proinflammatory cytokines', 'GeneOrGeneProduct'],
 [28428256, 'protein kinase a', 'GeneOrGeneProduct'],
 [20510337, 'lipid', 'ChemicalEntity'],
 [17975693, 'anxiogenic', 'DiseaseOrPhenotypicFeature'],
 [24743235, 'il-10', 'GeneOrGeneProduct'],
 [17192049, 'cytochrome p4501a1', 'GeneOrGeneProduct'],
 [18808529, 'oxygen', 'ChemicalEntity'],
 [20683499, 'neurodegenerative diseases', 'DiseaseOrPhenotypicFeature'],
 [28512644, 't2734c', 'SequenceVariant'],
 [24341598, 'calcium', 'ChemicalEntity'],
 [16737910, 'cancer', 'DiseaseOrPhenotypicFeature'],
 [28428256, 'inflammatory', 'DiseaseOrPhenotypicFeature'],
 [18827003, 'glucocorticoid', 'ChemicalEntity'],
 [16288197, 'mcor', 'GeneOrGeneProduct'],
 [15970799, 'slco1b1', 'GeneOrGeneProduct'],
 [29222418, 'vg4', 'GeneOrGeneProduct'],
 [18808529, 'myocardial injury', 'DiseaseOrPhenotypicFeature'],
 [25305591, 'anti-inflammatory cytokines', 'GeneOrGeneProduct'],
 [18827003, 'cortisol', 'ChemicalEntity'],
 [28411266, 'insulin', 'Gene